# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [51]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI




In [ ]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

In [10]:
links = fetch_website_links("https://vismaifood.com")
links = (list(set(links)))

In [11]:
links

['https://vismaifood.com/en/recipes/cuisine/indian',
 'https://in.pinterest.com/vismaifood/',
 'https://vismaifood.com/en/coconut-ven-pongal-recipe-kattu-pongali-recipe-coconut-pongal-recipe',
 'https://vismaifood.com/en/bhindi-do-pyaza-okra-onion-curry-easy-bhindi-do-pyaza-recipe-video-how-to-make-bhindi-do-pyaz',
 'https://vismaifood.com/en/recipes/summer-recipes',
 'https://vismaifood.com/en/recipes/cuisine/international',
 'https://vismaifood.com/en/recipes/baking',
 'https://vismaifood.com/en/recipes/starters',
 'https://vismaifood.com/en/recipes/wedding-style-recipes',
 'https://vismaifood.com/en/crispy-bhindi-kurkure',
 'https://vismaifood.com/en/recipes/cuisine/south-indian',
 'https://vismaifood.com/en/recipes/cuisine/kerala',
 'https://vismaifood.com/en/privacy-policy',
 'https://vismaifood.com/en/recipes/cuisine/indo-italian',
 'https://vismaifood.com/en/chatpata-desi-style-watermelon-slush-watermelon-slush-watermelon-recipe',
 'https://vismaifood.com/en/recipes/main-course'

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [13]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [14]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [16]:
print(get_links_user_prompt("https://vismaifood.com"))


Here is the list of links on the website https://vismaifood.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://vismaifood.com/en
javascript:void(0)
https://vismaifood.com/en/recipes
https://vismaifood.com/en/recipes
https://vismaifood.com/en/subscribe
https://vismaifood.com/en
#
https://vismaifood.com/en
https://vismaifood.com/en/recipes
https://vismaifood.com/en/subscribe
#
https://vismaifood.com/en/andhra-style-peethala-iguru-recipe-spicy-crab-curry-crab-recipe
https://vismaifood.com/en/andhra-style-peethala-iguru-recipe-spicy-crab-curry-crab-recipe
https://vismaifood.com/en/andhra-style-dried-anchovy-curry-raw-mango-mamidikaya-endu-nethallu-curry
https://vismaifood.com/en/andhra-style-dried-anchovy-curry-raw-mango-mamidikaya-endu-nethallu-curry
https://vismaifood.com/en/tirumala-tirupati-anna

In [27]:
def select_relevant_links(url):
    openai = OpenAI()
    response = openai.chat.completions.create(
        model="gpt-5-nano",
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [28]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


In [29]:
select_relevant_links("https://vismaifood.com")

{'links': [{'type': 'home page', 'url': 'https://vismaifood.com/en'},
  {'type': 'contact page', 'url': 'https://vismaifood.com/en/contact-us'}]}

In [30]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'blog', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter/X', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [ ]:
select_relevant_links("https://huggingface.co")

{'links': [{'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'company page', 'url': 'https://huggingface.co/huggingface'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'documentation hub', 'url': 'https://huggingface.co/docs'},
  {'type': 'learn page', 'url': 'https://huggingface.co/learn'},
  {'type': 'github', 'url': 'https://github.com/huggingface'},
  {'type': 'twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'discourse', 'url': 'https://discuss.huggingface.co'},
  {'type': 'status page', 'url': 'https://status.huggingface.co/'},
  {'type': 'endpoints page', 'url': 'https://endpoints.huggingface.co'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [35]:
print(fetch_website_contents("https://huggingface.co"))

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
HauhauCS/Qwen3.5-35B-A3B-Uncensored-HauhauCS-Aggressive
Updated
14 days ago
•
326k
•
897
Jackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled
Updated
about 20 hours ago
•
164k
•
1.17k
baidu/Qianfan-OCR
Updated
6 days ago
•
8.49k
•
334
nvidia/Nemotron-Cascade-2-30B-A3B
Updated
1 day ago
•
19.7k
•
257
RoyalCities/Foundation-1
Updated
8 days ago
•
248
Browse 2M+ models
Spaces
Running
on
Zero
MCP
1.53k
Wan2.2 14B Preview
🐌
1.53k
generate a video from an image with a text prompt
Running
on
Zero
Featured
167
DLSS 5 Anything
🎮
167
Turn any image into a DLSS 5 meme (using FLUX.2-klein-9b-kv)
Running
on
Zero
MCP
Featured
441
FireRed Image Edit 1.0 Fa

In [32]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [33]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
HauhauCS/Qwen3.5-35B-A3B-Uncensored-HauhauCS-Aggressive
Updated
14 days ago
•
326k
•
897
Jackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled
Updated
about 20 hours ago
•
164k
•
1.17k
baidu/Qianfan-OCR
Updated
6 days ago
•
8.49k
•
334
nvidia/Nemotron-Cascade-2-30B-A3B
Updated
1 day ago
•
19.7k
•
257
RoyalCities/Foundation-1
Updated
8 days ago
•
248
Browse 2M+ models
Spaces
Running
on
Zero
MCP
1.53k
Wan2.2 14B Preview
🐌
1.53k
generate a video from an image with a text prompt
Running
on
Zero
Featured
167
DLSS 5 Anything
🎮
167
Turn any image into a DLSS 5 meme (using FLUX.2-klein-9b-kv)
Running
on
Zero
MCP
Featured
441
FireRed

In [49]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information. And we want to attract the Genz people, Also add some light touch of humour to the brochure.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [37]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [39]:
print(get_brochure_user_prompt("HuggingFace", "https://huggingface.co"))


You are looking at a company called: HuggingFace
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.


## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
HauhauCS/Qwen3.5-35B-A3B-Uncensored-HauhauCS-Aggressive
Updated
14 days ago
•
326k
•
898
Jackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled
Updated
about 20 hours ago
•
164k
•
1.17k
baidu/Qianfan-OCR
Updated
6 days ago
•
8.49k
•
334
nvidia/Nemotron-Cascade-2-30B-A3B
Updated
1 minute ago
•
19.7k
•
257
RoyalCities/Foundation-1
Updated
8 days ago
•
248
Browse 2M+ models
Spaces
Running
on
Zero
MCP
1.53k
Wan2.2 14B 

In [43]:
def create_brochure(company_name, url):
    openai=OpenAI()
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [44]:
create_brochure("HuggingFace", "https://huggingface.co")

# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is the dynamic AI community and collaboration platform building the future of machine learning. It serves as a central hub where machine learning engineers, scientists, researchers, and end users come together to create, share, explore, and experiment with open-source ML models, datasets, and applications.

With over **2 million models** and more than **500,000 datasets**, Hugging Face accelerates AI innovation across modalities such as text, image, video, audio, and even 3D.

---

## What We Offer

### Collaborate and Build
- Host and collaborate on **unlimited public models, datasets, and applications**.
- Share your work publicly to build your ML portfolio and reputation.
- Discover trending and cutting-edge AI models updated regularly.
  
### Explore AI Applications
- Explore and run AI apps created by the community.
- Access featured spaces that showcase state-of-the-art applications like text-to-video generation, image editing, and more.
  
### Tools & Libraries
- Utilize some of the most popular open-source machine learning libraries and tools provided by Hugging Face.
- Accelerate your ML projects using the open-source HF stack.

### Enterprise Solutions
- Offers enterprise-grade services and solutions tailored to business needs (details accessible on the enterprise page).

---

## Community & Culture

- A **fast-growing, global collaborative community** dedicated to open and ethical AI.
- Encourages sharing, learning, and mutual growth for the next generation of machine learning professionals.
- Values transparency, openness, inclusivity, and innovation.
- Hugging Face’s talented science team pushes the boundaries of AI research and development.

---

## Our Customers

Hugging Face serves a wide spectrum of users including:
- Independent machine learning engineers and researchers.
- Developers and AI practitioners building advanced AI products.
- Enterprises seeking scalable AI deployment solutions.
- Academic institutions and hobbyists passionate about AI research.

---

## Careers at Hugging Face

Join a forward-thinking company at the heart of the AI revolution. Hugging Face offers:
- A vibrant, inclusive workplace where innovation is encouraged.
- Opportunities to work with cutting-edge ML technologies and contribute to open-source projects.
- Roles in AI research, engineering, product management, community, and more.

To explore current openings and join the community transforming AI, visit the [Careers page](https://huggingface.co/careers).

---

## Connect with Us

- Website: [huggingface.co](https://huggingface.co)
- GitHub: [github.com/huggingface](https://github.com/huggingface)
- Twitter: [@huggingface](https://twitter.com/huggingface)
- LinkedIn: [Hugging Face LinkedIn](https://www.linkedin.com/company/hugging-face/)
- Discord Community for real-time collaboration and support.

---

## Visual Identity

- Brand Colors: Bright yellow (#FFD21E), orange (#FF9D00), and gray (#6B7280).
- Logos and brand assets available in multiple formats (.svg, .png, .ai).

---

Join Hugging Face today — where the machine learning community builds the future!

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [47]:
def stream_brochure(company_name, url):
    openai=OpenAI()
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [48]:
stream_brochure("HuggingFace", "https://huggingface.co")

# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is a global AI community dedicated to building the future of machine learning (ML). As a leading collaboration platform, Hugging Face empowers ML engineers, scientists, and enthusiasts worldwide to explore, create, and share models, datasets, and applications in an open and ethical environment.

The platform hosts over **2 million models**, **500,000+ datasets**, and over **1 million applications**, supporting multiple modalities such as text, image, video, audio, and even 3D. Hugging Face’s ecosystem enables faster innovation by providing access to state-of-the-art open source ML tools, creating a central place for discovery, experimentation, and portfolio building.

---

## What We Offer

- **Model Hub:** Access and collaborate on millions of pre-trained and custom machine learning models for a variety of tasks, including natural language processing, computer vision, and video generation.
- **Datasets:** Explore and contribute to a vast collection of open datasets continuously updated by the community.
- **Spaces:** Run and share ML-powered applications and demos easily on hosted environments.
- **Open Source Stack:** Build faster and better with tools designed for community collaboration and scalability.
- **Collaboration Platform:** Host unlimited public models, datasets, and applications, with seamless community interaction.

---

## Company Culture

Hugging Face thrives on openness, inclusiveness, and collaboration. The company believes in:

- **Community First:** Empowering a global network to freely share and collaborate to accelerate the AI revolution.
- **Ethical AI:** Commitment to building transparent, responsible, and accessible AI technology that benefits everyone.
- **Innovation & Learning:** Continuous learning with an open platform that enables building and showcasing ML expertise.
- **Diversity & Inclusion:** Welcoming contributors from all backgrounds to foster diverse ideas and creative solutions.

---

## Our Customers & Community

Hugging Face serves a vibrant and rapidly growing community consisting of:

- Machine learning engineers and researchers pushing the boundaries of AI.
- Data scientists and developers building innovative AI applications.
- Enterprises leveraging AI models for business transformation.
- Educators and students advancing AI knowledge and education.
- Open source contributors enhancing the collective intelligence of AI models and datasets.

With thousands of organizations and individuals worldwide relying on Hugging Face for their AI workflows, the company is a cornerstone of the open AI ecosystem.

---

## Careers & Opportunities

Hugging Face is continuously expanding its team to support its mission of democratizing AI. Career opportunities include roles in:

- Machine Learning Research & Engineering
- Software Development and Infrastructure
- Data Science and Analytics
- Community Management and Developer Relations
- Product Management and Growth

The company offers a dynamic, supportive atmosphere where innovation, impact, and collaboration drive the work culture.

---

## Get Involved

Whether you are a researcher, developer, or business interested in cutting-edge AI, Hugging Face provides tools and community support to help you:

- **Create:** Build and publish your own ML models or datasets.
- **Discover:** Explore a vast range of state-of-the-art AI resources.
- **Collaborate:** Join forces with others on open projects.
- **Share:** Showcase your AI applications and build your professional portfolio.

Visit [Hugging Face](https://huggingface.co) to explore, join, or partner with the AI community building the future.

---

**Hugging Face — The Home of Machine Learning Innovation**  
Building Open, Ethical, and Accessible AI, Together.  
[Sign Up Today](https://huggingface.co/join) and become part of the revolution.

---

### Brand Colors 
- Hugging Face Yellow: #FFD21E  
- Orange: #FF9D00  
- Gray: #6B7280

### Logo Assets Available  
SVG, PNG, AI formats for versatile use across your projects.

---

**Contact & Support**  
Explore Documentation, Enterprise Solutions, Pricing, and more on the official site.  
Connect with a thriving AI ecosystem on Hugging Face.



In [50]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

# Welcome to Hugging Face  
*The AI community building the future — one model, one dataset, one collab at a time!*  

---

## Who We Are  
Hugging Face is the buzzing hive of machine learning enthusiasts globally — from curious rookies to seasoned AI wizards. Think of us as the ultimate playground where more than 2 million open-source models and over half a million datasets hang out, collaborate, and push the boundaries of AI innovation. Whether it's text, images, videos, audio, or even 3D, we've got **all** your AI modalities covered! 

Our mission? To empower the next generation of ML engineers, scientists, and creators (yep, we’re talking to you, Gen Z!) with tools and platforms designed to learn, share, and build an ethical AI future — without the boring corporate jargon.

---

## What We Offer  

### Hugging Face Hub  
- A central, one-stop shop to **browse, download, and contribute** to millions of AI models and datasets.  
- Collaborate freely with the community or showcase your own AI brainchildren.  
- Build your personal ML portfolio and become a recognized rockstar in the AI universe!  

### Spaces  
- Launch and share your AI-powered applications effortlessly.  
- From cool text-to-video apps to image-to-meme generators, experiment and impress your peers.  
- All running on zero hassle — because who likes slow-loading demos?  

### Open Source Stack  
- Move faster with our open source tools designed for you to **build, train, and deploy AI** better, quicker, and smarter.  

---

## Who Uses Hugging Face?  
- **AI researchers and engineers** looking for cutting-edge models and resources.  
- **Startups and enterprises** aiming to embed AI magic into their products without reinventing the wheel.  
- **Educators and students** hungry to learn and experiment with state-of-the-art machine learning.  
- **Creative coders and hobbyists** eager to turn ideas into AI-powered realities (yes, even memes count!).  

---

## Culture & Community  
At Hugging Face, we don’t just build AI — we build *human* connections. Our culture is all about:

- **Collaboration over competition**: It’s a team sport, baby!  
- **Open & ethical AI**: Building for good, with transparency and responsibility.  
- **Learning & sharing**: Because knowledge is the ultimate power-up.  
- **Fun and creativity**: Who says AI can’t have a little flair and humor?  

(We also believe a good emoji is worth a thousand words — just check out our Spaces where 🐌 and 🌖 live happily side by side.)  

---

## Careers: Join the Hugging Face Family  
Want to work where your ideas get to play with AI every single day? Hugging Face is scouting for bright minds who love to learn, build, and shape the future — all while having a bit of fun.  

- Positions for ML engineers, data scientists, devs, and community builders.  
- Remote and flexible work because your creativity doesn’t have a 9-5 schedule.  
- Competitive perks that’ll keep you fueled for coding marathons and meme creation at lunch breaks.  

If you have a passion for AI, openness, and a sprinkle of humor, we want you!  

---

## Ready to Get Started?  
Explore our models, datasets, and dazzling AI apps at [huggingface.co](https://huggingface.co)  
Build your future. Build it with Hugging Face.  

---

*Hugging Face: Where AI meets awesome — and you’re the star.*  
*(P.S. We promise not to replace your favorite memes with AI… yet.)*

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>